# Options Pricing Engine
## Black-Scholes | Binomial Tree | Monte Carlo

A clean implementation of the three canonical option pricing models,
with real market data integration via `yfinance`.

**Author:** Kanishka Sarkar  
**Stack:** Python · NumPy · SciPy · yfinance · Plotly

In [4]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from core import black_scholes as bs
from core import binomial_tree as bt
from core import monte_carlo as mc
from data.market_data import get_stock_data, get_risk_free_rate, time_to_expiry

print('Imports OK')

ModuleNotFoundError: No module named 'core'

## 1. Fetch live market data

In [ ]:
TICKER = 'AAPL'
data = get_stock_data(TICKER)
r = get_risk_free_rate()

print(f"Ticker:          {data['ticker']}")
print(f"Current price:   ${data['current_price']:.2f}")
print(f"Vol (30d):       {data['hist_volatility_30d']:.1%}")
print(f"Vol (252d):      {data['hist_volatility_252d']:.1%}")
print(f"Risk-free rate:  {r:.2%}")

## 2. Define option parameters

In [ ]:
S = data['current_price']          # current stock price
K = round(S, -1)                    # ATM strike (rounded to nearest 10)
T = 90 / 365                        # 90 days to expiry
sigma = data['hist_volatility_30d'] # 30-day historical vol as proxy
option_type = 'call'

print(f"S={S:.2f}, K={K:.2f}, T={T:.4f}y ({90}d), r={r:.2%}, σ={sigma:.1%}, type={option_type}")

## 3. Black-Scholes pricing

In [ ]:
bs_price_result = bs.price(S, K, T, r, sigma, option_type)
bs_greeks_result = bs.greeks(S, K, T, r, sigma, option_type)

print(f"BS Price: ${bs_price_result:.4f}")
print()
print("Greeks:")
for name, val in bs_greeks_result.items():
    print(f"  {name:<8} {val:.6f}")

## 4. Binomial Tree (CRR)

In [ ]:
euro_price = bt.price(S, K, T, r, sigma, option_type, style='european', steps=200)
amer_price = bt.price(S, K, T, r, sigma, option_type, style='american', steps=200)

print(f"Binomial (European): ${euro_price:.4f}")
print(f"Binomial (American): ${amer_price:.4f}")
print(f"Early exercise premium: ${amer_price - euro_price:.4f}")

# Convergence
steps_list = [5, 10, 25, 50, 100, 200, 300, 500]
prices_list = [bt.price(S, K, T, r, sigma, option_type, steps=n) for n in steps_list]

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps_list, y=prices_list, mode='lines+markers', name='Binomial'))
fig.add_hline(y=bs_price_result, line_dash='dash', annotation_text=f'BS=${bs_price_result:.4f}')
fig.update_layout(title='Binomial Convergence to Black-Scholes',
                  xaxis_title='Steps', yaxis_title='Price ($)')
fig.show()

## 5. Monte Carlo simulation

In [ ]:
mc_result = mc.price(S, K, T, r, sigma, option_type, n_simulations=100_000)
print(f"MC Price:    ${mc_result['price']:.4f}")
print(f"Std Error:   {mc_result['std_error']:.6f}")
print(f"95% CI:      [{mc_result['confidence_interval'][0]:.4f}, {mc_result['confidence_interval'][1]:.4f}]")

# Plot paths
times, paths = mc.simulate_paths(S, T, r, sigma, n_paths=100)

fig = go.Figure()
for i in range(50):
    fig.add_trace(go.Scatter(
        x=times * 365, y=paths[:, i],
        mode='lines', line=dict(width=0.5, color='rgba(99,110,250,0.3)'),
        showlegend=False
    ))
fig.add_hline(y=K, line_dash='dash', line_color='red', annotation_text=f'Strike K={K}')
fig.update_layout(title='GBM Simulated Price Paths', xaxis_title='Days', yaxis_title='Price ($)')
fig.show()

## 6. Model comparison & put-call parity

In [ ]:
print('=== Model Comparison ===')
print(f'Black-Scholes:   ${bs_price_result:.6f}')
print(f'Binomial (200):  ${euro_price:.6f}  (diff: {euro_price - bs_price_result:+.6f})')
print(f'Monte Carlo:     ${mc_result["price"]:.6f}  (diff: {mc_result["price"] - bs_price_result:+.6f})')

print()
print('=== Put-Call Parity ===')
c = bs.price(S, K, T, r, sigma, 'call')
p = bs.price(S, K, T, r, sigma, 'put')
lhs = c - p
rhs = S - K * np.exp(-r * T)
print(f'C - P = {lhs:.6f}')
print(f'S - Ke^(-rT) = {rhs:.6f}')
print(f'Error: {abs(lhs - rhs):.2e}  (should be ~0)')

## 7. Greeks visualisation

In [ ]:
spot_range = np.linspace(S * 0.5, S * 1.5, 100)
delta_vals = [bs.greeks(s, K, T, r, sigma, option_type)['delta'] for s in spot_range]
gamma_vals = [bs.greeks(s, K, T, r, sigma, option_type)['gamma'] for s in spot_range]

fig = make_subplots(rows=1, cols=2, subplot_titles=['Delta vs Spot', 'Gamma vs Spot'])
fig.add_trace(go.Scatter(x=spot_range, y=delta_vals, name='Delta', line=dict(color='royalblue')), row=1, col=1)
fig.add_trace(go.Scatter(x=spot_range, y=gamma_vals, name='Gamma', line=dict(color='tomato')), row=1, col=2)
for col in [1, 2]:
    fig.add_vline(x=K, line_dash='dash', line_color='gray', row=1, col=col)
fig.update_xaxes(title_text='Stock Price ($)')
fig.update_layout(title='Delta and Gamma vs Spot Price')
fig.show()